# 06 — Home Credit: Multi-Table Feature Engineering

The main table has **one row per applicant**, but each person also appears in several **child tables**
(past loans, previous applications, monthly payment records). XGBoost needs one row per person, so the
core skill is **aggregation**: collapse many child rows into summary features on the single application
row, with `groupby -> agg -> merge`.

We build features from two child tables (`bureau`, `previous_application`) and watch test AUC climb.

## 1. The main table and the one-to-many structure

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
HC = "../data/raw/home_credit"

app = pd.read_csv(f"{HC}/application_train.csv")
print("application_train:", app.shape, "| default rate", round(app.TARGET.mean(), 4))

bureau = pd.read_csv(f"{HC}/bureau.csv")
print("bureau:", bureau.shape, "(many past loans per applicant)")

# trace ONE applicant: 1 row in app, several rows in bureau
cid = bureau.groupby("SK_ID_CURR").size().pipe(lambda s: s[s == 5].index[0])
print(f"\nApplicant {cid}: TARGET =", app.loc[app.SK_ID_CURR==cid, "TARGET"].iloc[0])
print("Their bureau rows:")
print(bureau[bureau.SK_ID_CURR==cid][["SK_ID_BUREAU","CREDIT_ACTIVE","AMT_CREDIT_SUM","AMT_CREDIT_SUM_DEBT"]].to_string(index=False))

application_train: (307511, 122) | default rate 0.0807


bureau: (1716428, 17) (many past loans per applicant)

Applicant 100047: TARGET = 1
Their bureau rows:
 SK_ID_BUREAU CREDIT_ACTIVE  AMT_CREDIT_SUM  AMT_CREDIT_SUM_DEBT
      5357372        Closed        675000.0                  0.0
      5357373        Active        630000.0             692253.0
      5357374        Active       4045500.0                  NaN
      5357375        Closed       1479834.0                  0.0
      5357376        Active       2697300.0            2528203.5


## 2. The aggregation pattern (simple: 6 features)

`groupby(SK_ID_CURR)` buckets each applicant's rows; `.agg(name=(col, func))` collapses each bucket
into one number. To count a category (e.g. "Active"), make a 0/1 flag first, then sum it.

In [2]:
bureau["is_active"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
simple = bureau.groupby("SK_ID_CURR").agg(
    bureau_loan_count   = ("SK_ID_BUREAU", "count"),
    bureau_active_count = ("is_active", "sum"),
    bureau_total_debt   = ("AMT_CREDIT_SUM_DEBT", "sum"),
)
print("Applicant", cid, "collapsed to one row:")
print(simple.loc[cid].to_string())

Applicant 100047 collapsed to one row:
bureau_loan_count            5.0
bureau_active_count          3.0
bureau_total_debt      3220456.5


## 3. Richer bureau aggregation (~17 features)

Same pattern, but many columns each with several functions (sum/mean/max/min). More thorough
aggregation = more signal extracted from the same table.

In [3]:
bagg = bureau.groupby("SK_ID_CURR").agg({
    "SK_ID_BUREAU":["count"], "is_active":["sum","mean"],
    "AMT_CREDIT_SUM":["sum","mean","max"], "AMT_CREDIT_SUM_DEBT":["sum","mean","max"],
    "AMT_CREDIT_SUM_OVERDUE":["sum","max"], "CREDIT_DAY_OVERDUE":["max","mean"],
    "DAYS_CREDIT":["min","max","mean"], "CNT_CREDIT_PROLONG":["sum"]})
bagg.columns = ["bureau_"+c+"_"+f for c,f in bagg.columns]
print("bureau features:", bagg.shape[1])

bureau features: 17


## 4. A second child table: previous_application (~16 features)

The applicant's past applications with Home Credit itself, including how often they were approved or
refused. Your track record with *this* lender is especially predictive.

In [4]:
prev = pd.read_csv(f"{HC}/previous_application.csv")
prev["is_approved"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
prev["is_refused"]  = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
pagg = prev.groupby("SK_ID_CURR").agg({
    "SK_ID_PREV":["count"], "is_approved":["mean","sum"], "is_refused":["mean"],
    "AMT_APPLICATION":["mean","max","sum"], "AMT_CREDIT":["mean","max","sum"],
    "AMT_ANNUITY":["mean","max"], "DAYS_DECISION":["min","max"], "CNT_PAYMENT":["mean","sum"]})
pagg.columns = ["prev_"+c+"_"+f for c,f in pagg.columns]
print("previous_application features:", pagg.shape[1])

previous_application features: 16


## 5. Merge everything with a LEFT join

Left join keeps every applicant and attaches their features. Applicants with no history get `NaN`,
which XGBoost handles natively.

In [5]:
merged = app.merge(bagg, on="SK_ID_CURR", how="left").merge(pagg, on="SK_ID_CURR", how="left")
print("rows:", app.shape[0], "->", merged.shape[0], "(unchanged)  | columns:", merged.shape[1])
y = merged["TARGET"]
num = merged.select_dtypes("number").drop(columns=["TARGET","SK_ID_CURR"])
bcols = [c for c in num.columns if c.startswith("bureau_")]
pcols = [c for c in num.columns if c.startswith("prev_")]
base  = [c for c in num.columns if c not in bcols + pcols]
print("application numerics:", len(base), "| bureau:", len(bcols), "| prev:", len(pcols))

rows: 307511 -> 307511 (unchanged)  | columns: 155
application numerics: 104 | bureau: 17 | prev: 16


## 6. Does it compound? Staged AUC test

Same split and borrowers each time; only the feature set grows.

In [6]:
idx = np.arange(len(y))
i_tmp, i_test = train_test_split(idx, test_size=.2, stratify=y, random_state=42)
i_tr, i_val = train_test_split(i_tmp, test_size=.25, stratify=y.iloc[i_tmp], random_state=42)
spw = (y.iloc[i_tr]==0).sum() / (y.iloc[i_tr]==1).sum()

def run(cols, label):
    X = num[cols]
    m = XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=5, subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="auc",
        early_stopping_rounds=40, n_jobs=-1, random_state=42)
    m.fit(X.iloc[i_tr], y.iloc[i_tr], eval_set=[(X.iloc[i_val], y.iloc[i_val])], verbose=False)
    auc = roc_auc_score(y.iloc[i_test], m.predict_proba(X.iloc[i_test])[:, 1])
    print(f"  {label:42s} {len(cols):3d} feats   AUC = {auc:.4f}")
    return auc

a0 = run(base, "application only")
a1 = run(base + bcols, "+ bureau")
a2 = run(base + bcols + pcols, "+ bureau + previous_application")
print(f"\n  total lift from 2 child tables: +{a2-a0:.4f}")

  application only                           104 feats   AUC = 0.7542


  + bureau                                   121 feats   AUC = 0.7577


  + bureau + previous_application            137 feats   AUC = 0.7652

  total lift from 2 child tables: +0.0110


## 7. Recap
- Child tables are **one-to-many**; `groupby -> agg -> merge` collapses them to one row per applicant.
- To aggregate a category, make a **0/1 flag** then `sum`/`mean` it.
- **Left join** keeps all applicants; missing history becomes `NaN` (XGBoost handles it).
- Feature engineering **compounds**: richer aggregation + more tables kept raising AUC
  (~0.754 -> ~0.765 from just 2 of 6 tables). Extending to all 6 tables and hundreds of features is
  how the brief's 0.78+ target is reached. The technique is simple; **thoroughness** is the skill.